In [1]:
import pyodbc

In [2]:
def create_connection():
    # Thông tin kết nối SQL Server
    server = 'ZOHATEA'  # Tên server từ SSMS
    database = 'Customer'  # Đã sửa, bỏ dấu ;
    username = 'sa'
    password = 'ptit'

    # Kết nối SQL Server
    try:
        conn = pyodbc.connect(
            f"DRIVER={{ODBC Driver 17 for SQL Server}};"  # Sử dụng driver có sẵn
            f"SERVER={server};"
            f"DATABASE={database};"
            f"UID={username};"
            f"PWD={password}"
        )
        cursor = conn.cursor()
        print("Kết nối thành công!")
    except pyodbc.Error as ex:
        print(f"Lỗi kết nối: {ex}")
    return conn, cursor

# 1. Khach Hang

In [3]:
import pandas as pd

In [4]:
df_khachhang = pd.read_csv('dim_customer.csv', encoding='utf-8')
df_khachhang.to_csv('Customer/KhachHang.csv', index=False, encoding='utf-8')
df_khachhang

,customer_key,customerName,customerType,nationality
0,CUST00001,Vi Mai,Individual,Vietnam
1,CUST00002,김성수,Travel Agency,Korea
2,CUST00003,Kevin Brown,Corporate,Australia
3,CUST00004,Bảo Vũ,Corporate,Vietnam
4,CUST00005,Bà Lâm Bùi,Corporate,Vietnam
...,...,...,...,...
4995,CUST04996,An Hoàng,Corporate,Vietnam
4996,CUST04997,佐藤 加奈,Corporate,Japan
4997,CUST04998,Trung Phạm,Individual,Vietnam
4998,CUST04999,Ông Trung Vũ,VIP,Vietnam


# 2. DatPhong.

In [5]:
import random
from datetime import datetime, timedelta

In [6]:
# Hàm sinh ngẫu nhiên ngày trong khoảng năm 2024, 2025
def random_date(start, end):
    delta = end - start
    random_days = random.randint(0, delta.days)
    return start + timedelta(days=random_days)

In [7]:
start_date = datetime(2024, 1, 1)
end_date = datetime(2025, 7, 1)

booking_data = []
booking_id_counter = 1

In [8]:
for _, row in df_khachhang.iterrows():
    ma_kh = row["customer_key"]
    customer_type = row["customerType"]  # Corrected column name to match DataFrame

    if customer_type == "Individual":
        num_bookings = random.randint(1, 2)
    elif customer_type == "Corporate":
        num_bookings = random.randint(3, 6)
    elif customer_type == "Travel Agency":
        num_bookings = random.randint(5, 10)
    elif customer_type == "VIP":
        num_bookings = random.randint(8, 15)
    else:
        num_bookings = random.randint(1, 3)  # Fallback if data is missing

    for _ in range(num_bookings):
        #ma_dat_phong = f"DP{str(booking_id_counter).zfill(5)}"
        thoi_gian = random_date(start_date, end_date)
        ghi_chu = random.choice(["", "Yêu cầu phòng yên tĩnh", "Check-in sớm", "Có trẻ nhỏ", "Đặt giúp người khác"])

        booking_data.append({
           # "MaDatPhong": ma_dat_phong,
            "MaKH": ma_kh,
            "ThoiGian": thoi_gian.strftime("%Y-%m-%d"),
            "GhiChu": ghi_chu
        })
        booking_id_counter += 1

df_datphong = pd.DataFrame(booking_data)
df_datphong.sort_values(by="ThoiGian", inplace=True)
df_datphong


,MaKH,ThoiGian,GhiChu
12827,CUST02143,2024-01-01,
30237,CUST04902,2024-01-01,
19443,CUST03204,2024-01-01,Check-in sớm
12284,CUST02054,2024-01-01,
13688,CUST02277,2024-01-01,Check-in sớm
...,...,...,...
4927,CUST00825,2025-07-01,Check-in sớm
4430,CUST00747,2025-07-01,Yêu cầu phòng yên tĩnh
11213,CUST01866,2025-07-01,Yêu cầu phòng yên tĩnh
29968,CUST04852,2025-07-01,Check-in sớm


In [9]:
# tem Ma_dat_phong
df_datphong["MaDatPhong"] = ["DP" + str(i).zfill(5) for i in range(1, len(df_datphong) + 1)]
df_datphong = df_datphong[["MaDatPhong", "MaKH", "ThoiGian", "GhiChu"]]
df_datphong.to_csv('Customer/DatPhong.csv', index=False, encoding='utf-8')
df_datphong

,MaDatPhong,MaKH,ThoiGian,GhiChu
12827,DP00001,CUST02143,2024-01-01,
30237,DP00002,CUST04902,2024-01-01,
19443,DP00003,CUST03204,2024-01-01,Check-in sớm
12284,DP00004,CUST02054,2024-01-01,
13688,DP00005,CUST02277,2024-01-01,Check-in sớm
...,...,...,...,...
4927,DP30888,CUST00825,2025-07-01,Check-in sớm
4430,DP30889,CUST00747,2025-07-01,Yêu cầu phòng yên tĩnh
11213,DP30890,CUST01866,2025-07-01,Yêu cầu phòng yên tĩnh
29968,DP30891,CUST04852,2025-07-01,Check-in sớm


In [10]:
# lưu vao db
conn, cursor = create_connection()
try:
    cursor.executemany('''
    INSERT INTO DatPhong (MaDatPhong, MaKH, ThoiGian, GhiChu)
    VALUES (?, ?, ?, ?)
    ''', df_datphong.values.tolist())
    conn.commit()
    print(f"DatPhong: {len(df_datphong)} ")
except pyodbc.Error as ex:
    print(f"Lỗi khi chèn dữ liệu: {ex}")
    conn.rollback()
# Đóng kết nối  
conn.close()

Kết nối thành công!
Lỗi khi chèn dữ liệu: ('23000', '[23000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]The INSERT statement conflicted with the FOREIGN KEY constraint "FK__DatPhong__MaKH__267ABA7A". The conflict occurred in database "Customer", table "dbo.KhachHang", column \'MaKH\'. (547) (SQLExecDirectW); [23000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]The statement has been terminated. (3621)')


# 3. PhongDuocDat

In [11]:
import pandas as pd
import random
from datetime import datetime, timedelta

In [12]:
df_phong = pd.read_csv('Hotel/Phong.csv', encoding='utf-8')
phong_lich_su = {ma_phong: [] for ma_phong in df_phong["MaPhong"]}

In [13]:
def is_available(ma_phong, ngay_nhan, ngay_tra):
    for (bat_dau, ket_thuc) in phong_lich_su[ma_phong]:
        if not (ngay_tra <= bat_dau or ngay_nhan >= ket_thuc):
            return False
    return True

In [14]:
phong_duoc_dat = []
counter = 1

# Định nghĩa số đêm ở (ngẫu nhiên từ 1–5 đêm)
def random_date_range(start):
    so_dem = random.randint(1, 5)
    return start, start + timedelta(days=so_dem)

# Duyệt từng đơn đặt phòng
for _, row in df_datphong.iterrows():
    ma_dat_phong = row["MaDatPhong"]
    ngay_nhan_base = datetime.strptime(row["ThoiGian"], "%Y-%m-%d")

    # Số lượng phòng đặt cho đơn này: 1–3
    so_phong = random.choices([1, 2, 3], weights=[70, 25, 5])[0]

    so_da_chon = 0
    tried = 0
    max_try = 20  # Tránh vòng lặp vô tận

    while so_da_chon < so_phong and tried < max_try:
        ma_phong_row = df_phong.sample(1).iloc[0]
        ma_phong = ma_phong_row["MaPhong"]
        gia_niem_yet = ma_phong_row["GiaNiemYet"]

        ngay_nhan, ngay_tra = random_date_range(ngay_nhan_base)

        if is_available(ma_phong, ngay_nhan, ngay_tra):
            phong_lich_su[ma_phong].append((ngay_nhan, ngay_tra))
            tong_tien = (ngay_tra - ngay_nhan).days * gia_niem_yet

            phong_duoc_dat.append({
                "MaPhongDuocDat": f"PDD{str(counter).zfill(6)}",
                "MaDatPhong": ma_dat_phong,
                "MaPhong": ma_phong,
                "NgayNhanPhong": ngay_nhan.strftime("%Y-%m-%d"),
                "NgayTraPhong": ngay_tra.strftime("%Y-%m-%d"),
                "TongTienPhong": tong_tien
            })
            counter += 1
            so_da_chon += 1
        tried += 1


In [15]:
df_phongduocdat = pd.DataFrame(phong_duoc_dat)
df_phongduocdat.to_csv('Customer/PhongDuocDat.csv', index=False, encoding='utf-8')
df_phongduocdat

,MaPhongDuocDat,MaDatPhong,MaPhong,NgayNhanPhong,NgayTraPhong,TongTienPhong
0,PDD000001,DP00001,THA001_R019,2024-01-01,2024-01-06,8500000
1,PDD000002,DP00002,BRV004_R007,2024-01-01,2024-01-03,2400000
2,PDD000003,DP00003,GLI001_R017,2024-01-01,2024-01-02,1800000
3,PDD000004,DP00003,BDH002_R021,2024-01-01,2024-01-03,4000000
4,PDD000005,DP00004,DLK002_R001,2024-01-01,2024-01-02,1200000
...,...,...,...,...,...,...
41621,PDD041622,DP30889,QNG001_R024,2025-07-01,2025-07-05,11200000
41622,PDD041623,DP30890,BTN001_R014,2025-07-01,2025-07-06,11000000
41623,PDD041624,DP30891,TBI002_R013,2025-07-01,2025-07-02,2400000
41624,PDD041625,DP30892,PYN002_R020,2025-07-01,2025-07-03,3000000


In [16]:
# lưu vao db
conn, cursor = create_connection()
try:
    cursor.executemany('''
    INSERT INTO PhongDuocDat (MaPhongDuocDat, MaDatPhong, MaPhong, NgayNhanPhong, NgayTraPhong, TongTienPhong)
    VALUES (?, ?, ?, ?, ?, ?)
    ''', df_phongduocdat.values.tolist())
    conn.commit()
    print(f"PhongDuocDat: {len(df_phongduocdat)} ")
except pyodbc.Error as ex:
    print(f"Lỗi khi chèn dữ liệu: {ex}")
    conn.rollback()
# Đóng kết nối  
conn.close()

Kết nối thành công!
Lỗi khi chèn dữ liệu: ('23000', '[23000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]The INSERT statement conflicted with the FOREIGN KEY constraint "FK__PhongDuoc__MaDat__29572725". The conflict occurred in database "Customer", table "dbo.DatPhong", column \'MaDatPhong\'. (547) (SQLExecDirectW); [23000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]The statement has been terminated. (3621)')


# 4. DichVu

In [17]:
df_dichvu = pd.read_csv('Hotel/DichVu.csv', encoding='utf-8')
df_dichvu


,MaDichVu,TenDichVu,LoaiDichVu,DonGia,MoTa
0,SVC001,Ăn uống trong phòng,Dịch vụ phòng,300000,"Set menu bữa sáng/tối, phục vụ 24/24."
1,SVC002,Quầy bar mini,Dịch vụ phòng,120000,"Nước ngọt, bia, snack (tính theo món)."
2,SVC003,Dọn phòng hàng ngày,Dịch vụ phòng,0,Đã bao gồm trong giá phòng.
3,SVC004,Liệu trình Spa,Chăm sóc sức khỏe,1200000,Gói cơ bản 90 phút (massage body + mặt nạ).
4,SVC005,Trị liệu massage,Chăm sóc sức khỏe,800000,Massage toàn thân 60 phút.
5,SVC006,Xông hơi,Chăm sóc sức khỏe,300000,Sử dụng phòng xông hơi 1 lượt (tối đa 2 giờ).
6,SVC007,Lớp Yoga,Chăm sóc sức khỏe,200000,"Lớp nhóm 60 phút, giá/buổi."
7,SVC008,Sử dụng hồ bơi,Giải trí,0,Miễn phí cho khách lưu trú.
8,SVC009,Trung tâm thể hình,Giải trí,0,"Mở cửa 6h-22h, miễn phí thiết bị."
9,SVC010,Sân tennis,Giải trí,200000,"Thuê sân 1 giờ (bao gồm vợt, bóng)."


In [18]:

# Tạo dict tra cứu thời gian nhận/trả cho từng phòng
phong_time_dict = {
    row["MaPhongDuocDat"]: (row["NgayNhanPhong"], row["NgayTraPhong"])
    for _, row in df_phongduocdat.iterrows()
}

# Tạo dict tra cứu đơn giá theo MaDichVu
gia_dv_dict = {
    row["MaDichVu"]: row["DonGia"]
    for _, row in df_dichvu.iterrows()
}

# Tạo danh sách tất cả các MaDichVu có sẵn
ds_dichvu_san = list(gia_dv_dict.keys())

dich_vu_sudung_data = []
counter = 1

for ma_pdd, (nhan_str, tra_str) in phong_time_dict.items():
    # Số dịch vụ sử dụng cho phòng này: 0–5 lần
    so_lan = random.randint(0, 5)
    
    for _ in range(so_lan):
        ma_dv = random.choice(ds_dichvu_san)
        don_gia = gia_dv_dict[ma_dv]
        so_luong = random.randint(1, 3)

        # Sinh thời gian ngẫu nhiên trong khoảng ở
        ngay_nhan = datetime.strptime(nhan_str, "%Y-%m-%d")
        ngay_tra = datetime.strptime(tra_str, "%Y-%m-%d") - timedelta(days=1)
        if ngay_nhan > ngay_tra:
            thoi_gian = ngay_nhan
        else:
            thoi_gian = ngay_nhan + timedelta(days=random.randint(0, (ngay_tra - ngay_nhan).days))

        dich_vu_sudung_data.append({
            #"MaSuDungDichVu": f"SDV{str(counter).zfill(6)}",
            "MaDichVu": ma_dv,
            "MaPhongDuocDat": ma_pdd,
            "ThoiGian": thoi_gian.strftime("%Y-%m-%d"),
            "SoLuong": so_luong,
            "ThanhTien": so_luong * don_gia
        })
        counter += 1


In [19]:
df_sudung_dichvu = pd.DataFrame(dich_vu_sudung_data)
df_sudung_dichvu.sort_values(by="ThoiGian", inplace=True)

df_sudung_dichvu["MaSuDungDichVu"] = ["SDV" + str(i).zfill(6) for i in range(1, len(df_sudung_dichvu) + 1)]
df_sudung_dichvu = df_sudung_dichvu[["MaSuDungDichVu", "MaDichVu", "MaPhongDuocDat", "ThoiGian", "SoLuong", "ThanhTien"]]

df_sudung_dichvu.to_csv('Customer/DichVu.csv', index=False, encoding='utf-8')

df_sudung_dichvu

,MaSuDungDichVu,MaDichVu,MaPhongDuocDat,ThoiGian,SoLuong,ThanhTien
0,SDV000001,SVC001,PDD000001,2024-01-01,3,900000
103,SDV000002,SVC025,PDD000039,2024-01-01,1,250000
102,SDV000003,SVC010,PDD000039,2024-01-01,2,400000
101,SDV000004,SVC005,PDD000039,2024-01-01,3,2400000
100,SDV000005,SVC006,PDD000039,2024-01-01,2,600000
...,...,...,...,...,...,...
104269,SDV104320,SVC017,PDD041605,2025-07-04,1,500000
104108,SDV104321,SVC009,PDD041536,2025-07-04,2,0
104263,SDV104322,SVC002,PDD041603,2025-07-04,3,360000
104314,SDV104323,SVC023,PDD041623,2025-07-05,2,600000


# 5. Dánh giá

In [20]:
nhan_xet_theo_diem = {
    5: ["Tuyệt vời!", "Dịch vụ hoàn hảo.", "Rất hài lòng, sẽ quay lại."],
    4: ["Tốt, nhưng còn vài điểm cần cải thiện.", "Nhân viên thân thiện, phòng sạch sẽ."],
    3: ["Tạm ổn, cần nâng cấp một số tiện nghi.", "Phục vụ trung bình, giá hợp lý."],
    2: ["Không hài lòng lắm.", "Phòng hơi bẩn, phục vụ chưa tốt."],
    1: ["Rất tệ, không quay lại.", "Trải nghiệm tồi tệ."]
}


In [21]:
df_phongduocdat

,MaPhongDuocDat,MaDatPhong,MaPhong,NgayNhanPhong,NgayTraPhong,TongTienPhong
0,PDD000001,DP00001,THA001_R019,2024-01-01,2024-01-06,8500000
1,PDD000002,DP00002,BRV004_R007,2024-01-01,2024-01-03,2400000
2,PDD000003,DP00003,GLI001_R017,2024-01-01,2024-01-02,1800000
3,PDD000004,DP00003,BDH002_R021,2024-01-01,2024-01-03,4000000
4,PDD000005,DP00004,DLK002_R001,2024-01-01,2024-01-02,1200000
...,...,...,...,...,...,...
41621,PDD041622,DP30889,QNG001_R024,2025-07-01,2025-07-05,11200000
41622,PDD041623,DP30890,BTN001_R014,2025-07-01,2025-07-06,11000000
41623,PDD041624,DP30891,TBI002_R013,2025-07-01,2025-07-02,2400000
41624,PDD041625,DP30892,PYN002_R020,2025-07-01,2025-07-03,3000000


In [22]:

nhan_xet_theo_diem = {
    5: ["Tuyệt vời!", "Dịch vụ hoàn hảo.", "Rất hài lòng, sẽ quay lại."],
    4: ["Tốt, nhưng còn vài điểm cần cải thiện.", "Nhân viên thân thiện, phòng sạch sẽ."],
    3: ["Tạm ổn, cần nâng cấp một số tiện nghi.", "Phục vụ trung bình, giá hợp lý."],
    2: ["Không hài lòng lắm.", "Phòng hơi bẩn, phục vụ chưa tốt."],
    1: ["Rất tệ, không quay lại.", "Trải nghiệm tồi tệ."]
}

danhgia_data = []

for _, row in df_phongduocdat.iterrows():
    ma_pdd = row["MaPhongDuocDat"]
    ngay_tra = datetime.strptime(row["NgayTraPhong"], "%Y-%m-%d")

    if random.random() < 0.7:
        # Lần đánh giá thứ nhất
        thoi_gian_1 = ngay_tra + timedelta(days=random.randint(1, 3))
        diem_1 = random.choices([5, 4, 3, 2, 1], weights=[0.2, 0.5, 0.2, 0.08, 0.02])[0]
        nhan_xet_1 = random.choice(nhan_xet_theo_diem[diem_1])

        danhgia_data.append({
            "MaDanhGia": f"DG{ma_pdd}_1",
            "MaPhongDuocDat": ma_pdd,
            "ThoiGian": thoi_gian_1.strftime("%Y-%m-%d"),
            "DiemDanhGia": diem_1,
            "NhanXet": nhan_xet_1
        })

        # Có thể có lần đánh giá thứ hai (chỉnh sửa)
        if random.random() < 0.3:  # ~30% có chỉnh sửa
            thoi_gian_2 = thoi_gian_1 + timedelta(days=random.randint(1, 2))
            diem_2 = random.choices([5, 4, 3, 2, 1], weights=[0.2, 0.5, 0.2, 0.08, 0.02])[0]
            nhan_xet_2 = random.choice(nhan_xet_theo_diem[diem_2])

            danhgia_data.append({
                "MaDanhGia": f"DG{ma_pdd}_2",
                "MaPhongDuocDat": ma_pdd,
                "ThoiGian": thoi_gian_2.strftime("%Y-%m-%d"),
                "DiemDanhGia": diem_2,
                "NhanXet": nhan_xet_2
            })

In [23]:
df_danhgia = pd.DataFrame(danhgia_data)
# sort theo ThoiGian
df_danhgia.sort_values(by="ThoiGian", inplace=True)
df_danhgia.to_csv('Customer/DanhGia.csv', index=False, encoding='utf-8')
df_danhgia

,MaDanhGia,MaPhongDuocDat,ThoiGian,DiemDanhGia,NhanXet
3,DGPDD000005_1,PDD000005,2024-01-03,4,"Tốt, nhưng còn vài điểm cần cải thiện."
20,DGPDD000021_1,PDD000021,2024-01-03,4,"Nhân viên thân thiện, phòng sạch sẽ."
58,DGPDD000062_1,PDD000062,2024-01-03,5,"Rất hài lòng, sẽ quay lại."
46,DGPDD000046_1,PDD000046,2024-01-03,3,"Phục vụ trung bình, giá hợp lý."
53,DGPDD000055_1,PDD000055,2024-01-04,3,"Tạm ổn, cần nâng cấp một số tiện nghi."
...,...,...,...,...,...
37695,DGPDD041605_1,PDD041605,2025-07-08,3,"Phục vụ trung bình, giá hợp lý."
37713,DGPDD041626_2,PDD041626,2025-07-08,4,"Nhân viên thân thiện, phòng sạch sẽ."
37680,DGPDD041591_1,PDD041591,2025-07-09,5,Dịch vụ hoàn hảo.
37705,DGPDD041619_1,PDD041619,2025-07-09,3,"Tạm ổn, cần nâng cấp một số tiện nghi."
